# Task 0: Dataset Construction
This task builds the training corpus of 1000 Indian names, cleans each name, and stores the final list in TrainingNames.txt. The output is used as the character-level supervision set for all sequence models.

In [ ]:
# Core numerical and deep-learning stack used across all models in this notebook.
import torch  # Tensor operations and autograd
import torch.nn as nn  # Neural network layers
import torch.optim as optim  # Optimizers (Adam, SGD, etc.)
import torch.nn.functional as F  # Functional ops like pad/softmax
import numpy as np  # Numerical helpers
import random  # Python RNG
import math
import string

# Fix all random generators to make training and sampling reproducible.
SEED = 42  # Fixed seed for reproducible results
torch.manual_seed(SEED)  # Seed PyTorch RNG
np.random.seed(SEED)  # Seed NumPy RNG
random.seed(SEED)  # Seed Python RNG

# Select CUDA automatically when available to speed up tensor operations.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use GPU if available
print(f"Using device: {device}")

Using device: cpu


In [ ]:
# %pip install -q indian-names

# Data collection dependencies: regex cleaning + name source package.
import re
import random
import indian_names

# Keep random name sampling deterministic for reproducible datasets.
random.seed(42)  # Keep generated dataset deterministic


# Clean and validate each sampled name before adding to corpus.
def clean_name(s):
    # Remove digits/punctuation/symbols and trim any leftover whitespace.
    s = re.sub(r"[^A-Za-z]", "", str(s)).strip()  # Remove non-letters and trim whitespace
    # Enforce simple quality bounds so names are realistic and trainable.
    if len(s) < 3 or len(s) > 15:  # Drop names that are too short/long
        return ""
    return s.lower()  # Normalize case for consistent vocabulary


# Build unique name set for character-level training.
names_set = set()  # Set keeps only unique names
max_tries = 20000  # Safety cap to avoid infinite loop
tries = 0

# Repeatedly sample until we have 1000 unique valid names.
while len(names_set) < 1000 and tries < max_tries:
    tries += 1
    g = "male" if random.random() < 0.5 else "female"  # Balance gender sources
    nm = clean_name(indian_names.get_first_name(gender=g))  # Generate and clean
    if nm:
        names_set.add(nm)  # Add only valid cleaned names

# Explicit guard if source package returns too many invalid/repeated names.
if len(names_set) < 1000:
    raise ValueError(f"Could collect only {len(names_set)} unique names. Re-run cell.")

# Sort names so saved file order is stable across runs.
names = sorted(list(names_set))[:1000]  # Sort to keep saved order deterministic

# Persist dataset to disk (one name per line).
with open("TrainingNames.txt", "w", encoding="utf-8") as f:
    for n in names:
        f.write(n + "\n")  # Save one name per line

print(f"Saved {len(names)} real Indian names to TrainingNames.txt")
print("Sample:", names[:20])

Saved 1000 real Indian names to TrainingNames.txt
Sample: ['aanchal', 'aaradhita', 'aarohi', 'aarti', 'aastha', 'abha', 'abhay', 'abhigyan', 'abhijeet', 'abhilash', 'abhilasha', 'abhilesh', 'abhinav', 'abhisar', 'abhishek', 'adhya', 'aditi', 'aditya', 'adityanath', 'adya']


In [ ]:
# Special symbols used by sequence models.
SOS_TOKEN = '<'  # Start-of-sequence token
EOS_TOKEN = '>'  # End-of-sequence token
PAD_TOKEN = '#'  # Padding token for equal sequence lengths

# Load cleaned training names.
with open("TrainingNames.txt", 'r', encoding='utf-8') as f:
    names = [line.strip().lower() for line in f.readlines()]  # Read and normalize names

# Build character vocabulary from dataset + special tokens.
chars = set(''.join(names))  # Collect unique characters from dataset
chars.add(SOS_TOKEN)
chars.add(EOS_TOKEN)
chars.add(PAD_TOKEN)

# Integer encoding maps required for embedding lookup and decoding.
char_to_idx = {char: idx for idx, char in enumerate(sorted(list(chars)))}  # Char -> integer index
idx_to_char = {idx: char for char, idx in char_to_idx.items()}  # Integer index -> char
vocab_size = len(char_to_idx)

# Shared hyperparameters used by all three recurrent architectures.
EMBEDDING_DIM = 48
HIDDEN_DIM = 128
NUM_LAYERS = 2
LEARNING_RATE = 0.002
EPOCHS = 50
BATCH_SIZE = 64


# Convert a single name string to token indices with boundary markers.
def name_to_tensor(name):
    indices = [char_to_idx[SOS_TOKEN]] + [char_to_idx[c] for c in name] + [char_to_idx[EOS_TOKEN]]
    return torch.tensor(indices, dtype=torch.long)  # Convert token indices to tensor


# Longest sequence length (+2 accounts for SOS/EOS tokens).
max_len = max([len(n) for n in names]) + 2  # +2 for SOS/EOS


# Right-pad sequence to fixed length for batched training.
def pad_sequence(tensor, max_length):
    pad_len = max_length - len(tensor)
    return F.pad(tensor, (0, pad_len), value=char_to_idx[PAD_TOKEN])  # Right-pad to fixed length


# Build full dataset matrix shape [num_names, max_len].
dataset = [pad_sequence(name_to_tensor(n), max_len) for n in names]  # Tokenize and pad every name
dataset = torch.stack(dataset).to(device)  # Stack into [N, T] tensor and move to device

print(f"Dataset Size: {dataset.shape}")
print(f"Vocabulary Size: {vocab_size}")

Dataset Size: torch.Size([1000, 16])
Vocabulary Size: 27


In [ ]:
print("Sample tensor:", dataset[1])  # Inspect one tokenized+padded name
print(f"SOS token index: {char_to_idx[SOS_TOKEN]}")  # Verify SOS mapping
print(f"EOS token index: {char_to_idx[EOS_TOKEN]}")  # Verify EOS mapping
print(f"PAD token index: {char_to_idx[PAD_TOKEN]}")  # Verify PAD mapping

Sample tensor: tensor([ 1,  3,  3, 19,  3,  6, 10, 11, 21,  3,  2,  0,  0,  0,  0,  0])
SOS token index: 1
EOS token index: 2
PAD token index: 0


# Task 1: Model Implementation (From Scratch)
In this task, three recurrent architectures are implemented with custom model classes and explicit training loops: Vanilla RNN, BLSTM, and RNN with additive attention. For each model, architecture, parameter count, and hyperparameters are reported.

### Model 1: Vanilla RNN
A simple autoregressive baseline with character embeddings, recurrent hidden state updates, and linear token prediction at each time step.

In [ ]:
# Vanilla character-level RNN implemented with explicit recurrent parameters.
class VanillaCharRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1):
        super(VanillaCharRNN, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Embedding converts token indices to dense vectors for recurrent processing.
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=char_to_idx[PAD_TOKEN]  # PAD embedding is ignored during training
        )

        # Each layer stores explicit RNN weights (no nn.RNN used).
        self.rnn_layers = nn.ModuleList()  # Hold per-layer RNN parameters
        for layer_idx in range(num_layers):
            layer_input_dim = embedding_dim if layer_idx == 0 else hidden_dim  # First layer uses embeddings
            layer_params = nn.ParameterDict({
                'W_xh': nn.Parameter(torch.randn(hidden_dim, layer_input_dim) * 0.01),  # Input -> hidden
                'W_hh': nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01),  # Hidden -> hidden
                'b_h': nn.Parameter(torch.zeros(hidden_dim))  # Hidden bias
            })
            self.rnn_layers.append(layer_params)

        # Final projection from hidden state to vocabulary logits.
        self.fc = nn.Linear(in_features=hidden_dim, out_features=vocab_size)  # Hidden -> vocab logits

    def forward(self, x, hidden=None):
        batch_size, seq_len = x.shape
        device = x.device
        embedded = self.embedding(x)  # [B, T, E]

        # If not provided, start hidden states at zeros for all layers.
        if hidden is None:
            hidden = torch.zeros(
                self.num_layers, batch_size, self.hidden_dim,
                dtype=embedded.dtype, device=device
            )  # Initialize hidden state for all layers

        outputs = []
        # Iterate autoregressively across sequence positions.
        for t in range(seq_len):
            x_t = embedded[:, t, :]  # Current token embedding
            for layer_idx in range(self.num_layers):
                params = self.rnn_layers[layer_idx]
                W_xh, W_hh, b_h = params['W_xh'], params['W_hh'], params['b_h']
                h_prev = hidden[layer_idx]

                # Core update: h_t = tanh(W_xh x_t + W_hh h_prev + b).
                h_new = torch.tanh(torch.matmul(x_t, W_xh.t()) + torch.matmul(h_prev, W_hh.t()) + b_h)  # RNN update
                hidden[layer_idx] = h_new  # Store state for next timestep
                x_t = h_new  # Feed to next stacked layer

            outputs.append(hidden[-1])  # Top-layer state is output at timestep t

        rnn_output = torch.stack(outputs, dim=1)  # [B, T, H]
        logits = self.fc(rnn_output)  # [B, T, V]
        return logits, hidden


# Instantiate baseline model and report complexity.
model_rnn = VanillaCharRNN(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS).to(device)
params_rnn = sum(p.numel() for p in model_rnn.parameters() if p.requires_grad)  # Count trainable params
print(f"Vanilla RNN - Trainable Parameters: {params_rnn}")
print("Architecture: Embedding -> Custom RNN (from scratch) -> Linear(vocab)")
print(f"Hyperparameters: embedding_dim={EMBEDDING_DIM}, hidden_dim={HIDDEN_DIM}, num_layers={NUM_LAYERS}, lr={LEARNING_RATE}")


# Shared teacher-forcing training loop used for autoregressive models.
def train_autoregressive(model, data, epochs=EPOCHS):
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss(ignore_index=char_to_idx[PAD_TOKEN])  # Ignore padded targets
    model.train()

    for epoch in range(epochs):
        optimizer.zero_grad()

        # Input is all tokens except last; target is shifted by one step.
        inputs = data[:, :-1]  # Teacher-forcing input tokens
        targets = data[:, 1:]  # Next-token labels
        logits, _ = model(inputs)

        # Flatten batch/time dims so CE sees [N, vocab] vs [N].
        loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))  # Flatten [B,T] for CE
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")


print("Training Vanilla RNN...")
train_autoregressive(model_rnn, dataset)

Vanilla RNN - Trainable Parameters: 60587
Architecture: Embedding -> RNN -> Linear(vocab)
Hyperparameters: hidden_dim=128, num_layers=2, lr=0.002
Training Vanilla RNN...
Epoch 10/50, Loss: 2.4002
Epoch 20/50, Loss: 2.1518
Epoch 30/50, Loss: 2.0284
Epoch 40/50, Loss: 1.9460
Epoch 50/50, Loss: 1.8799


In [ ]:
import os

params_rnn = sum(p.numel() for p in model_rnn.parameters() if p.requires_grad)  # Total trainable params
print(f"Vanilla RNN trainable parameters: {params_rnn:,}")

# Estimate memory footprint from number of parameters and dtype size.
bytes_per_param = next(model_rnn.parameters()).element_size()  # Usually 4 bytes for float32
size_mb_theoretical = (params_rnn * bytes_per_param) / (1024 ** 2)  # Approx in-memory model size
print(f"Theoretical model size: {size_mb_theoretical:.4f} MB")

# Save only learned weights (state_dict), not full Python object.
save_path = "vanilla_rnn_state_dict.pt"
torch.save(model_rnn.state_dict(), save_path)  # Save learned weights only
size_mb_saved = os.path.getsize(save_path) / (1024 ** 2)  # Actual serialized file size
print(f"Saved state_dict size: {size_mb_saved:.4f} MB at {save_path}")

Vanilla RNN trainable parameters: 60,587
Theoretical model size: 0.2311 MB
Saved state_dict size: 0.2355 MB at vanilla_rnn_state_dict.pt


### Model 2: Bidirectional LSTM (BLSTM)
A stronger recurrent baseline that captures both left and right context before projecting to vocabulary logits.

In [ ]:
# Single LSTM cell with explicit gate parameters.
class LSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(LSTMCell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        self.W_ii = nn.Parameter(torch.randn(hidden_dim, input_dim) * 0.01)  # Input gate: input weights
        self.W_hi = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)  # Input gate: recurrent weights
        self.b_i = nn.Parameter(torch.zeros(hidden_dim))  # Input gate bias

        self.W_if = nn.Parameter(torch.randn(hidden_dim, input_dim) * 0.01)  # Forget gate: input weights
        self.W_hf = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)  # Forget gate: recurrent weights
        self.b_f = nn.Parameter(torch.ones(hidden_dim) * 0.1)  # Slight positive bias to encourage remembering initially

        self.W_ig = nn.Parameter(torch.randn(hidden_dim, input_dim) * 0.01)  # Candidate gate: input weights
        self.W_hg = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)  # Candidate gate: recurrent weights
        self.b_g = nn.Parameter(torch.zeros(hidden_dim))  # Candidate bias

        self.W_io = nn.Parameter(torch.randn(hidden_dim, input_dim) * 0.01)  # Output gate: input weights
        self.W_ho = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)  # Output gate: recurrent weights
        self.b_o = nn.Parameter(torch.zeros(hidden_dim))  # Output gate bias

    def forward(self, x_t, h_t_prev, c_t_prev):
        # Gate computations controlling write/forget/expose behavior.
        i_t = torch.sigmoid(torch.matmul(x_t, self.W_ii.t()) + torch.matmul(h_t_prev, self.W_hi.t()) + self.b_i)  # How much new info to write
        f_t = torch.sigmoid(torch.matmul(x_t, self.W_if.t()) + torch.matmul(h_t_prev, self.W_hf.t()) + self.b_f)  # How much old memory to keep
        g_t = torch.tanh(torch.matmul(x_t, self.W_ig.t()) + torch.matmul(h_t_prev, self.W_hg.t()) + self.b_g)  # Candidate memory values
        o_t = torch.sigmoid(torch.matmul(x_t, self.W_io.t()) + torch.matmul(h_t_prev, self.W_ho.t()) + self.b_o)  # How much memory to expose

        c_t = f_t * c_t_prev + i_t * g_t  # Update cell memory
        h_t = o_t * torch.tanh(c_t)  # Produce hidden state

        return h_t, c_t


# Bidirectional LSTM stacks forward/backward LSTM streams per layer.
class BLSTMCharModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1):
        super(BLSTMCharModel, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=char_to_idx[PAD_TOKEN]
        )  # Map chars to dense vectors

        self.forward_lstm_layers = nn.ModuleList()  # Left-to-right cells
        self.backward_lstm_layers = nn.ModuleList()  # Right-to-left cells

        for layer_idx in range(num_layers):
            layer_input_dim = embedding_dim if layer_idx == 0 else (hidden_dim * 2)  # Upper layers consume concatenated bi-output
            self.forward_lstm_layers.append(LSTMCell(layer_input_dim, hidden_dim))
            self.backward_lstm_layers.append(LSTMCell(layer_input_dim, hidden_dim))

        self.fc = nn.Linear(in_features=hidden_dim * 2, out_features=vocab_size)  # Bi-hidden -> vocab logits

    def forward(self, x, hidden=None):
        batch_size, seq_len = x.shape
        device = x.device
        embedded = self.embedding(x)  # [B, T, E]

        # Hidden tensors store [layer, direction, batch, hidden_dim].
        if hidden is None:
            h_states = torch.zeros(self.num_layers, 2, batch_size, self.hidden_dim, dtype=embedded.dtype, device=device)  # Hidden states per layer/direction
            c_states = torch.zeros(self.num_layers, 2, batch_size, self.hidden_dim, dtype=embedded.dtype, device=device)  # Cell states per layer/direction
        else:
            h_states, c_states = hidden

        for layer_idx in range(self.num_layers):
            forward_cell = self.forward_lstm_layers[layer_idx]
            backward_cell = self.backward_lstm_layers[layer_idx]

            h_fwd = h_states[layer_idx, 0]
            c_fwd = c_states[layer_idx, 0]
            h_bwd = h_states[layer_idx, 1]
            c_bwd = c_states[layer_idx, 1]

            # Forward temporal pass.
            forward_outputs = []
            for t in range(seq_len):
                x_t = embedded[:, t, :] if layer_idx == 0 else layer_outputs[t]  # Use embeddings for first layer
                h_fwd, c_fwd = forward_cell(x_t, h_fwd, c_fwd)
                forward_outputs.append(h_fwd)

            # Backward temporal pass.
            backward_outputs = []
            for t in range(seq_len - 1, -1, -1):
                x_t = embedded[:, t, :] if layer_idx == 0 else layer_outputs[t]
                h_bwd, c_bwd = backward_cell(x_t, h_bwd, c_bwd)
                backward_outputs.append(h_bwd)
            backward_outputs.reverse()  # Realign to original left-to-right order

            h_states[layer_idx, 0] = h_fwd
            c_states[layer_idx, 0] = c_fwd
            h_states[layer_idx, 1] = h_bwd
            c_states[layer_idx, 1] = c_bwd

            # Concatenate directions and feed next stacked layer.
            fwd = torch.stack(forward_outputs, dim=1)  # [B, T, H]
            bwd = torch.stack(backward_outputs, dim=1)  # [B, T, H]
            embedded = torch.cat((fwd, bwd), dim=2)  # [B, T, 2H]
            layer_outputs = [embedded[:, t, :] for t in range(seq_len)]  # Prepare per-step inputs for next layer

        logits = self.fc(embedded)
        return logits, (h_states, c_states)


# Instantiate BLSTM and train with same autoregressive objective for fair comparison.
model_blstm = BLSTMCharModel(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS).to(device)
params_blstm = sum(p.numel() for p in model_blstm.parameters() if p.requires_grad)
print(f"Bidirectional LSTM - Trainable Parameters: {params_blstm}")
print("Architecture: Embedding -> Custom Bidirectional LSTM (from scratch) -> Linear(vocab)")
print(f"Hyperparameters: embedding_dim={EMBEDDING_DIM}, hidden_dim={HIDDEN_DIM}, num_layers={NUM_LAYERS}, lr={LEARNING_RATE}")

print("Training Bidirectional LSTM...")
train_autoregressive(model_blstm, dataset)

BLSTM - Trainable Parameters: 585771
Architecture: Embedding -> Bidirectional LSTM -> Linear(vocab)
Hyperparameters: hidden_dim=128, num_layers=2, lr=0.002
Training BLSTM...
Epoch 10/50, Loss: 2.4911
Epoch 20/50, Loss: 1.8634
Epoch 30/50, Loss: 1.0990
Epoch 40/50, Loss: 0.4430
Epoch 50/50, Loss: 0.1447


### Model 3: RNN with Basic Additive Attention
An encoder-decoder style model where additive attention dynamically selects useful encoder states at each decoding step.

In [ ]:
# Minimal RNN cell used by both encoder and decoder in attention model.
class SimpleRNNCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(SimpleRNNCell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        self.W_xh = nn.Parameter(torch.randn(hidden_dim, input_dim) * 0.01)  # Input-to-hidden weights
        self.W_hh = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)  # Hidden-to-hidden weights
        self.b_h = nn.Parameter(torch.zeros(hidden_dim))  # Hidden bias

    def forward(self, x_t, h_prev):
        h_t = torch.tanh(torch.matmul(x_t, self.W_xh.t()) + torch.matmul(h_prev, self.W_hh.t()) + self.b_h)  # Standard RNN update
        return h_t


# Seq2Seq model with bidirectional encoder and additive attention decoder.
class Seq2SeqAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(Seq2SeqAttention, self).__init__()

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.pad_idx = char_to_idx[PAD_TOKEN]

        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=self.pad_idx)  # Shared embedding

        self.encoder_forward = SimpleRNNCell(embedding_dim, hidden_dim)  # Left-to-right encoder
        self.encoder_backward = SimpleRNNCell(embedding_dim, hidden_dim)  # Right-to-left encoder

        # Additive attention projection layers.
        self.attn_weight = nn.Linear(in_features=hidden_dim * 3, out_features=hidden_dim)  # Project [decoder;encoder]
        self.attn_score = nn.Linear(in_features=hidden_dim, out_features=1, bias=False)  # Scalar attention score

        self.decoder = SimpleRNNCell(embedding_dim + hidden_dim * 2, hidden_dim)  # Decoder takes token embedding + context
        self.output_proj = nn.Linear(in_features=hidden_dim, out_features=vocab_size)  # Hidden -> vocab logits

    def forward(self, src, trg):
        batch_size = src.size(0)
        src_len = src.size(1)
        trg_len = trg.size(1)
        device = src.device

        # PAD mask ensures attention ignores padded source positions.
        src_mask = (src != self.pad_idx).float()  # 1 for real tokens, 0 for PAD
        embedded_src = self.embedding(src)  # [B, S, E]

        # Encode source sequence in forward direction.
        encoded_fwd = []
        h_fwd = torch.zeros(batch_size, self.hidden_dim, device=device, dtype=embedded_src.dtype)  # Forward encoder state
        for t in range(src_len):
            h_fwd = self.encoder_forward(embedded_src[:, t, :], h_fwd)
            encoded_fwd.append(h_fwd)

        # Encode source sequence in backward direction.
        encoded_bwd = []
        h_bwd = torch.zeros(batch_size, self.hidden_dim, device=device, dtype=embedded_src.dtype)  # Backward encoder state
        for t in range(src_len - 1, -1, -1):
            h_bwd = self.encoder_backward(embedded_src[:, t, :], h_bwd)
            encoded_bwd.append(h_bwd)
        encoded_bwd.reverse()  # Align backward states to source order

        encoder_outputs = torch.cat([torch.stack(encoded_fwd, dim=1), torch.stack(encoded_bwd, dim=1)], dim=2)  # [B, S, 2H]
        h_dec = (h_fwd + h_bwd) / 2  # Initialize decoder state from both directions

        embedded_trg = self.embedding(trg)  # [B, T, E]
        outputs = torch.zeros(batch_size, trg_len, self.vocab_size, device=device, dtype=embedded_src.dtype)  # Store logits per step

        # Decode each target timestep with dynamic attention context.
        for t in range(trg_len):
            h_dec_expanded = h_dec.unsqueeze(1).expand(-1, src_len, -1)  # Repeat decoder state across source positions
            attention_input = torch.cat((h_dec_expanded, encoder_outputs), dim=2)  # Combine decoder + encoder info
            attention_features = torch.tanh(self.attn_weight(attention_input))  # Nonlinear attention features
            attention_scores = self.attn_score(attention_features).squeeze(2)  # Unnormalized scores over source positions

            attention_scores = attention_scores.masked_fill(src_mask == 0, -1e9)  # Block PAD from receiving attention
            attention_weights = F.softmax(attention_scores, dim=1)  # Normalize to probabilities
            context = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs).squeeze(1)  # Weighted sum context vector

            current_embed = embedded_trg[:, t, :]  # Teacher-forced decoder token at time t
            decoder_input = torch.cat((current_embed, context), dim=1)  # Feed token + context to decoder
            h_dec = self.decoder(decoder_input, h_dec)  # Update decoder state
            outputs[:, t, :] = self.output_proj(h_dec)  # Predict next-token logits

        return outputs


# Teacher-forcing training loop for sequence-to-sequence attention model.
def train_seq2seq(model, data, epochs=EPOCHS):
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss(ignore_index=char_to_idx[PAD_TOKEN])  # Ignore PAD targets
    model.train()

    for epoch in range(epochs):
        optimizer.zero_grad(set_to_none=True)

        src = data  # Encoder sees full sequence
        trg_input = data[:, :-1]  # Decoder input (teacher forcing)
        trg_target = data[:, 1:]  # Ground-truth next tokens

        logits = model(src, trg_input)
        loss = criterion(logits.reshape(-1, vocab_size), trg_target.reshape(-1))  # Flatten for CE
        loss.backward()

        # Clip gradients for stability on long recurrent computations.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Prevent exploding gradients
        optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")


# Instantiate attention model and train.
model_attn = Seq2SeqAttention(vocab_size, EMBEDDING_DIM, HIDDEN_DIM).to(device)
params_attn = sum(p.numel() for p in model_attn.parameters() if p.requires_grad)
print(f"RNN with Attention (Custom) - Trainable Parameters: {params_attn}")
print("Architecture: Embedding -> Bidirectional RNN Encoder (custom) -> Additive Attention -> RNN Decoder (custom) -> Linear(vocab)")
print(f"Hyperparameters: embedding_dim={EMBEDDING_DIM}, hidden_dim={HIDDEN_DIM}, attention=additive, lr={LEARNING_RATE}")

print("Training RNN with Attention...")
train_seq2seq(model_attn, dataset)

RNN with Attention - Trainable Parameters: 155307
Architecture: RNN encoder -> additive attention -> RNN decoder -> Linear(vocab)
Hyperparameters: hidden_dim=128, num_layers=1, lr=0.002
Training Seq2Seq with Attention...
Epoch 10/50, Loss: 2.3530
Epoch 20/50, Loss: 1.8475
Epoch 30/50, Loss: 1.3508
Epoch 40/50, Loss: 0.9496
Epoch 50/50, Loss: 0.4645


# Task 2: Quantitative Evaluation
This task evaluates generated names using novelty rate and diversity so the three architectures can be compared with the same generation budget.

In [ ]:
# Sample from autoregressive models (Vanilla RNN / BLSTM).
def generate_from_autoregressive(model, num_names=200):
    model.eval()  # Switch to inference mode
    generated = []
    with torch.no_grad():  # Disable gradients for faster generation
        for _ in range(num_names):
            input_seq = torch.tensor([[char_to_idx[SOS_TOKEN]]]).to(device)  # Start with SOS
            hidden = None
            name = ""
            for _ in range(15):  # Hard max length
                logits, hidden = model(input_seq, hidden)
                probs = F.softmax(logits[:, -1, :] / 0.8, dim=-1)  # Temperature sampling (0.8)
                next_char_idx = torch.multinomial(probs, 1).item()  # Sample next character
                if next_char_idx == char_to_idx[EOS_TOKEN]:
                    break  # Stop when EOS is produced
                if next_char_idx != char_to_idx[PAD_TOKEN]:
                    name += idx_to_char[next_char_idx]  # Append valid character
                input_seq = torch.tensor([[next_char_idx]]).to(device)  # Feed predicted token back
            if len(name) > 2:
                generated.append(name.capitalize())  # Keep only plausible names
    return generated


# Sample from seq2seq attention model by growing decoder input token by token.
def generate_from_seq2seq(model, num_names=200):
    model.eval()  # Switch to inference mode
    generated = []
    with torch.no_grad():  # Disable gradients for faster generation
        for _ in range(num_names):
            rand_idx = random.randint(0, len(dataset) - 1)
            src = dataset[rand_idx].unsqueeze(0)  # Pick one source sequence as encoder context
            trg = torch.tensor([[char_to_idx[SOS_TOKEN]]]).to(device)  # Decoder starts with SOS
            name = ""
            for _ in range(15):  # Hard max length
                logits = model(src, trg)
                probs = F.softmax(logits[:, -1, :] / 0.8, dim=-1)  # Temperature sampling
                next_char_idx = torch.multinomial(probs, 1).item()  # Sample next token
                if next_char_idx == char_to_idx[EOS_TOKEN]:
                    break
                if next_char_idx != char_to_idx[PAD_TOKEN]:
                    name += idx_to_char[next_char_idx]
                trg = torch.cat((trg, torch.tensor([[next_char_idx]]).to(device)), dim=1)  # Extend decoder input
            if len(name) > 2:
                generated.append(name.capitalize())
    return generated


# Use identical generation budget for fair model comparison.
print("Generating evaluation samples...")
gen_rnn = generate_from_autoregressive(model_rnn, 500)  # 500 samples from Vanilla RNN
gen_blstm = generate_from_autoregressive(model_blstm, 500)  # 500 samples from BLSTM
gen_attn = generate_from_seq2seq(model_attn, 500)  # 500 samples from attention model

# Lowercased train-set lookup for novelty computation.
train_set = set([n.lower() for n in names])  # Training set for novelty check


# Compute novelty/diversity metrics from generated outputs.
def evaluate(generated_list, model_name):
    if len(generated_list) == 0:
        return

    novel = [n for n in generated_list if n.lower() not in train_set]  # Names not seen in training data
    novelty_rate = len(novel) / len(generated_list)

    unique_names = set(generated_list)  # Unique generated samples
    diversity = len(unique_names) / len(generated_list)

    print(f"--- {model_name} ---")
    print(f"Novelty Rate: {novelty_rate * 100:.2f}%")
    print(f"Diversity:    {diversity * 100:.2f}%\n")


# Print quantitative comparison for all models.
evaluate(gen_rnn, "Vanilla RNN")
evaluate(gen_blstm, "BLSTM")
evaluate(gen_attn, "RNN with Attention")

Generating evaluation samples...
--- Vanilla RNN ---
Novelty Rate: 97.19%
Diversity:    99.40%

--- BLSTM ---
Novelty Rate: 100.00%
Diversity:    73.79%

--- RNN with Attention ---
Novelty Rate: 75.75%
Diversity:    95.39%



# Task 3: Qualitative Analysis
This section prints representative names from each model to inspect realism, repetition artifacts, and stylistic differences that are not captured by metrics alone.

In [ ]:
# Show random sample names for qualitative inspection.
print("Representative Generated Samples:\n")  # Qualitative side-by-side comparison
print(f"Vanilla RNN: \n{random.sample(gen_rnn, min(10, len(gen_rnn)))}\n")  # Show up to 10 samples
print(f"BLSTM: \n{random.sample(gen_blstm, min(10, len(gen_blstm)))}\n")  # Show up to 10 samples
print(f"RNN w/ Attention: \n{random.sample(gen_attn, min(10, len(gen_attn)))}\n")  # Show up to 10 samples

Representative Generated Samples:

Vanilla RNN: 
['Ansa', 'Rashithar', 'Prayat', 'Nari', 'Dildap', 'Saneravi', 'Parshita', 'Areli', 'Hurina', 'Saminda']

BLSTM: 
['Uuuuuuuuuuuuuuu', 'Jvegeeeeeeeeeee', 'Ggggggggggggggg', 'Uauauauyuyyyyyy', 'Kakakakakakakak', 'Uuuuuuuuuuuuuuu', 'Gjgjjjjgjjjjjjo', 'Nknkkkkkkkkkkkk', 'Kakakkkkkkkkkyk', 'Padadadadadadad']

RNN w/ Attention: 
['Rushita', 'Sharanaji', 'Mona', 'Rathana', 'Drantadyi', 'Prajjslal', 'Bharav', 'Vilac', 'Rajkudra', 'Ishika']

